In [1]:
import os
import pandas as pd
import logging

In [2]:
import csv
import json

from src.schemas import TA_logprob_list, WordImportance
from src.utils.word_analyzer import WordAnalyzerBuilder

In [3]:
from pydantic import BaseModel

In [4]:
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True
)

In [5]:
logger = logging.getLogger("words_e")

In [6]:
logger.debug("test")

2026-04-23 21:19:10,268 - DEBUG - test


In [7]:
class Args(BaseModel):
    stage: str
    input: str
    token_metric: str
    word_metric: str
    

args = Args(
    stage="../data/samples_3/test2/",
    input="../data/samples_3/",
    token_metric="topk_token_entropy",
    word_metric="first",
)

In [8]:
in_checks = os.path.join(args.stage, "checks.csv")
in_logprobs = os.path.join(args.stage, "gen_logprobs.csv")
in_passages = os.path.join(args.input, "passages.json")

# файлик с контекстами
with open(in_passages, "r") as f:
    passages = json.load(f)
logger.info(f"Readed {in_passages} passages {len(passages)}")


# датасет с колонками
# ["question", "answer", "passage_id", "similarity"]
# индекс "check_id"
checks_df = pd.read_csv(in_checks, quoting=csv.QUOTE_NONNUMERIC, index_col=0)
checks_df.index = checks_df.index.astype(int)
checks_df["passage_id"] = checks_df["passage_id"].astype(int).astype(str)
logger.info(f"Checks readed from {in_checks} shape: {checks_df.shape}")

# датасет с колонками
# ["prompt_logprobs", "gen_answer"]
# индекс "check_id"
logprobs_df = pd.read_csv(in_logprobs, quoting=csv.QUOTE_NONNUMERIC, index_col=0)
logprobs_df.index = logprobs_df.index.astype(int)
logger.info(
    f"Logprobs dataset readed from {in_logprobs} shape: {logprobs_df.shape}"
)

2026-03-10 21:30:36,158 - INFO - Readed ../data/samples_3/passages.json passages 3
2026-03-10 21:30:36,163 - INFO - Checks readed from ../data/samples_3/test2/checks.csv shape: (44, 4)
2026-03-10 21:30:36,237 - INFO - Logprobs dataset readed from ../data/samples_3/test2/gen_logprobs.csv shape: (44, 2)


In [9]:
logprobs_j = logprobs_df.loc[9259,"prompt_logprobs"]
logprobs = TA_logprob_list.validate_json(logprobs_j)

In [10]:
logprobs

[None,
 {'872': PromptLogprob(decoded_token='user', logprob=-10.38395881652832, rank=4056),
  '67': PromptLogprob(decoded_token='d', logprob=-7.1964592933654785, rank=1),
  '82': PromptLogprob(decoded_token='s', logprob=-7.5714592933654785, rank=2),
  '258': PromptLogprob(decoded_token='in', logprob=-7.6027092933654785, rank=3),
  '15136': PromptLogprob(decoded_token='times', logprob=-7.6027092933654785, rank=4),
  '8': PromptLogprob(decoded_token=')', logprob=-7.6652092933654785, rank=5)},
 {'198': PromptLogprob(decoded_token='\n', logprob=-0.14488349854946136, rank=1),
  '271': PromptLogprob(decoded_token='\n\n', logprob=-2.269883394241333, rank=2),
  '25': PromptLogprob(decoded_token=':', logprob=-4.144883632659912, rank=3),
  '2610': PromptLogprob(decoded_token='You', logprob=-5.144883632659912, rank=4),
  '40': PromptLogprob(decoded_token='I', logprob=-6.644883632659912, rank=5)},
 {'2147': PromptLogprob(decoded_token='context', logprob=-23.620494842529297, rank=68146),
  '2610': 

In [11]:
builder = WordAnalyzerBuilder()
builder.set_token_metric(args.token_metric)
builder.set_word_metric(args.word_metric)
analyzer = builder.build()


2026-03-10 21:30:36,343 - INFO - Builded WordAnalyzer with metadata: {"token_metric": "topk_token_entropy", "word_metric": "first"}


In [12]:
builder.metadata

{'token_metric': 'topk_token_entropy', 'word_metric': 'first'}

In [13]:
tokens_importances = analyzer.tokens_importances(prompt_logprobs=logprobs)

In [14]:
passage_id = checks_df.loc[9259,"passage_id"]

In [15]:
passages[passage_id]

'(1) Спустя какое-то время везение прекращается. (2) Редакции наперебой стараются обжулить Мартина. (3) Добыть у них деньги за публикации оказывается нелёгким делом. (4) Руфь настаивает на том, чтобы Мартин устроился на работу к её отцу, она не верит в то, что он станет писателем. (5) Случайно у Морзов Мартин знакомится с Рэссом Бриссенденом и близко сходится с ним. (6) Бриссенден болен чахоткой, он не боится смерти, но страстно любит жизнь во всех её проявлениях. (7) Бриссенден знакомит Мартина с «настоящими людьми», одержимыми литературой и философией. (8) Со своим новым товарищем Мартин посещает митинг социалистов, где спорит с оратором, но благодаря расторопному и нещепетильному репортёру попадает на страницы газет в качестве социалиста и ниспровергателя существующего строя. (9) Газетная публикация приводит к печальным последствиям — Руфь присылает Мартину письмо, извещающее о разрыве помолвки. (10) Мартин продолжает жить по инерции, и его даже не радуют поступающие от журналов чек

In [17]:
words_importances = analyzer.words_importances(
    tokens_importances=tokens_importances,
    passage=passages[passage_id]
)

In [25]:
from operator import itemgetter

def modify_text_and_get_indices(text: str, replacements: list[WordImportance]):
    # Сортируем список замен по начальному индексу (слева направо)
    # Это обязательно, чтобы смещение работало корректно
    replacements.sort(key=itemgetter("start"))
    
    modified_text = text
    new_positions: list[WordImportance] = []  # Здесь будем хранить новые координаты
    offset: int = 0          # Текущее смещение индексов
    
    for replacement in replacements:
        # 1. Вычисляем точную позицию начала в текущей (уже частично измененной) строке
        actual_start = replacement["start"] + offset
        
        # 2. Вычисляем новый конец (это просто начало + длина нового слова)
        actual_end = actual_start + len(replacement["word"])

        new_word = replacement["word"]
        
        # 3. Сохраняем информацию о новом положении слова
        new_positions.append({
            'word': new_word,
            'start': actual_start,
            'end': actual_end,
            'importance': replacement["importance"]
        })
        
        # 4. Производим замену в строке
        # modified_text[end + offset:] берет остаток строки после старого слова
        modified_text = modified_text[:actual_start] + new_word + modified_text[replacement["end"] + offset:]
        
        # 5. Пересчитываем смещение для следующих слов
        # Разница между длиной вставленного и вырезанного фрагментов
        offset += len(new_word) - (replacement["end"] - replacement["start"])
        
    return modified_text, new_positions

# --- Проверяем работу ---

original_text = "Мама мыла раму мылом"
# Индексы: "мыла" (5:9), "мылом" (15:20)
changes = [
    {
        'word': "краской",
        'start': 15,
        'end': 20,
        'importance': -1.0
    },
    {
        'word': "очень долго красила",
        'start': 5,
        'end': 9,
        'importance': -1.0
    }
]

new_text, new_indices = modify_text_and_get_indices(original_text, changes)

print(f"Исходная строка: '{original_text}'")
print(f"Новая строка:    '{new_text}'\n")

print("Новые позиции слов:")
for item in new_indices:
    # Проверяем срезом, что индексы верные
    extracted_word = new_text[item['start']:item['end']]
    print(f"Слово '{item['word']}' теперь находится с {item['start']} по {item['end']} индекс. (Проверка: '{extracted_word}')")

Исходная строка: 'Мама мыла раму мылом'
Новая строка:    'Мама очень долго красила раму краской'

Новые позиции слов:
Слово 'очень долго красила' теперь находится с 5 по 24 индекс. (Проверка: 'очень долго красила')
Слово 'краской' теперь находится с 30 по 37 индекс. (Проверка: 'краской')


In [26]:
new_indices

[{'word': 'очень долго красила', 'start': 5, 'end': 24, 'importance': -1.0},
 {'word': 'краской', 'start': 30, 'end': 37, 'importance': -1.0}]